# AdaptCLIP ViSA Result Table

ViSA 전체 클래스에 대해 AdaptCLIP image-level AUROC와 top score를 저장합니다.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "models").is_dir() and (path / "datasets").is_dir() and (path / "utils").is_dir():
            return path
    raise RuntimeError("Repository root was not found from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


In [ ]:
import csv
import os

import pandas as pd
from sklearn.metrics import roc_auc_score

import config_visa
from datasets.visa import ViSA
from models.adaptclip import AdaptCLIP


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)


In [ ]:
visa_root = REPO_ROOT / "data" / "Visa"
categories = [
    folder.name
    for folder in visa_root.iterdir()
    if folder.is_dir() and folder.name != "split_csv"
]

categories


In [ ]:
results = []

for name in categories:
    train_data = ViSA(
        name,
        phase="Normal",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TRAIN_LIMIT,
    )

    test_normal = ViSA(
        name,
        phase="Normal",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TEST_LIMIT,
    )

    test_anomaly = ViSA(
        name,
        phase="Anomaly",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TEST_LIMIT,
    )

    adaptclip = AdaptCLIP(
        category=name,
        device=device,
        checkpoint_path="visa",
        img_resize=336,
        img_cropsize=336,
        batch_size=config_visa.BATCH_SIZE,
        use_prompt_query=True,
    )
    adaptclip.fit(train_data)

    scores = []
    labels = []
    for test_data in [test_normal, test_anomaly]:
        for img, label in test_data:
            score, _ = adaptclip.predict(img)
            scores.append(score.item())
            labels.append(int(label))

    image_auc = roc_auc_score(labels, scores)
    top_score = max(scores) if scores else float("nan")

    results.append([
        name,
        config_visa.TRAIN_LIMIT,
        "visa",
        round(image_auc, 4),
        round(top_score, 4),
    ])
    print(name, "AUROC", image_auc)

results = sorted(results, key=lambda x: x[3], reverse=True)

file_path = "adaptclip_visa_results.csv"
with open(file_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["category", "train_limit", "checkpoint", "auc", "top_score"])
    writer.writerows(results)

file_path


In [ ]:
df = pd.read_csv("adaptclip_visa_results.csv")
df
